In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("data/raw/train_data.csv")

# Convert 'severity' column to numerical (0 for Low, 1 for High)
df['severity'] = df['severity'].map({'Low': 0, 'High': 1})

# 2. Text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['grievance_text'] = df['grievance_text'].apply(clean_text)

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['grievance_text'], df['severity'], test_size=0.2, random_state=42
)

# 4. Tokenization
vocab_size = 10000
max_len = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# 5. Build BiRNN Model
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 6. Compile model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# 7. Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3)

# 8. Train model
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

# 9. Evaluate
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", accuracy)

# 10. Save model
model.save("outputs/models/bilstm_complaint_model.h5")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
270/270 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 13s 25ms/step - accuracy: 0.9775 - loss: 0.0634 - val_accuracy: 0.9995 - val_loss: 7.1640e-04
Epoch 2/10
270/270 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 6s 22ms/step - accuracy: 1.0000 - loss: 8.1092e-05 - val_accuracy: 1.0000 - val_loss: 4.4263e-05
Epoch 3/10
270/270 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 6s 23ms/step - accuracy: 1.0000 - loss: 3.0777e-05 - val_accuracy: 1.0000 - val_loss: 2.1230e-05
Epoch 4/10
270/270 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 5s 20ms/step - accuracy: 1.0000 - loss: 1.6263e-05 - val_accuracy: 1.0000 - val_loss: 1.2356e-05
Epoch 5/10
270/270 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 6s 23ms/step - accuracy: 1.0000 - loss: 9.9756e-06 - val_accuracy: 1.0000 - val_loss: 8.7420e-06
Epoch 6/10
270/270 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 5s 20ms/step - accuracy: 1.0000 - loss

Test Accuracy: 1.0


In [5]:
import pandas as pd
import re
import tensorflow as tf
#from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# Load trained model
model = tf.keras.models.load_model("outputs/models/bilstm_complaint_model.h5")

# Load tokenizer (IMPORTANT: tokenizer must be saved during training)
with open("outputs/models/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

max_len = 100

# Text cleaning function (same as training)
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

# Load new test file
df = pd.read_excel("data/raw/test_data.xlsx")

# Clean text
df['grievance_text'] = df['grievance_text'].apply(clean_text)

# Convert text â†’ sequence
seq = tokenizer.texts_to_sequences(df['grievance_text'])

# Padding
pad = pad_sequences(seq, maxlen=max_len, padding='post')

# Predict probabilities
probs = model.predict(pad)

# Convert to class using threshold 0.5
df['predicted_probability'] = probs
df['predicted_class'] = df['predicted_probability'].apply(
    lambda x: "High" if x >= 0.5 else "Low"
)

# Save output file
df.to_csv("data/processed/predicted_complaints_output.csv", index=False)

print("Output file generated: data/processed/predicted_complaints_output.csv")

721/721 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 6s 7ms/step
Output file generated: data/processed/predicted_complaints_output.csv


In [3]:
import pickle

# Save tokenizer
with open("outputs/models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved")

Tokenizer saved


In [21]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
import pickle
from keras.utils import pad_sequences

# Load model
model = tf.keras.models.load_model('outputs/models/bilstm_complaint_model.h5')

# Load tokenizer
with open('outputs/models/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

max_len = 100

# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

# Load test file
df = pd.read_excel('data/raw/Book1_english.xlsx')

# Keep only grievance_text (which has a trailing space initially)
df = df[['grievance_text ']]

# Rename the column to remove the trailing space
df.rename(columns={'grievance_text ': 'grievance_text'}, inplace=True)

# Clean text
df['grievance_text'] = df['grievance_text'].apply(clean_text)

# Text â†’ sequence
seq = tokenizer.texts_to_sequences(df['grievance_text'])

# Padding
pad = pad_sequences(seq, maxlen=max_len, padding='post')

# Predict probabilities
probs = model.predict(pad)

# Add results to dataframe
df['predicted_probability'] = probs
df['predicted_class'] = df['predicted_probability'].apply(
    lambda x: 'High' if x >= 0.3 else 'Low'
)

# Save output
df.to_csv('data/processed/predicted_output1.csv', index=False)

print("Prediction file saved as predicted_output.csv")

1563/1563 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 90s 57ms/step
Prediction file saved as predicted_output.csv
